# Preparación en Google Colab
Esta versión descarga los módulos y documentos públicos del repositorio para evitar tener que subir carpetas manualmente. Ejecute la preparación y configure la conexión **antes de grabar**. La clave privada solo se utiliza en la memoria de la sesión; nunca la escriba en el código ni la publique.

La sustentación muestra el notebook y brevemente Qdrant. Elegimos Qdrant porque está expresamente permitido en la guía, ofrece una instancia gestionada gratuita y permite consultar vectores con metadatos y distancia coseno.


In [3]:
# Ejecutar una vez al abrir una sesión nueva de Google Colab.
import sys, subprocess, urllib.request, os
from pathlib import Path
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'fastembed==0.9.1', 'qdrant-client==1.20.0',
                'python-dotenv==1.2.4', 'pandas', 'PyMuPDF'], check=True)
ROOT = Path('/content/ACA1_Nube')
for folder in ['src','corpus','data','results']: (ROOT/folder).mkdir(parents=True, exist_ok=True)
base = 'https://raw.githubusercontent.com/OverCometa/aca1-nube-busqueda-semantica/main/'
files = ['src/pipeline.py','corpus/manifest.json', 'corpus/01_guia_academica.md',
         'corpus/02_guia_financiera.md','corpus/03_guia_bienestar.md','data/queries.json']
for name in files:
    urllib.request.urlretrieve(base + name, ROOT/name)
os.chdir(ROOT)
print('Proyecto descargado. Continúe con la conexión privada.')


Proyecto descargado. Continúe con la conexión privada.


## Conexión privada
Ejecute esta celda antes de grabar. Necesita la API key del clúster; el campo no muestra sus caracteres.

In [4]:
# La clave se introduce en un campo oculto. No queda escrita en el código.
import os, getpass
os.environ['QDRANT_URL'] = 'https://c27e7b77-bedd-438c-9f88-e5fe6e2dc10d.sa-east-1-0.aws.cloud.qdrant.io'
if not os.environ.get('QDRANT_API_KEY'):
    os.environ['QDRANT_API_KEY'] = getpass.getpass('Clave privada de Qdrant (entrada oculta): ')
print('Credencial preparada para esta sesión; su valor no se muestra.')


Credencial preparada para esta sesión; su valor no se muestra.


# Consulta semántica cloud para orientación sobre permanencia estudiantil
**ACA 1 · Computación en la Nube · CUN**  
**Over Santiago Cometa Hernández · Trabajo individual · Octubre de 2026**

## Problema y objetivo
¿Cómo recuperar fragmentos pertinentes para preguntas sobre acompañamiento académico, apoyos económicos y bienestar, conservando trazabilidad y metadatos?

Se implementa ingesta, fragmentación, embeddings abiertos multilingües, carga por lotes y recuperación Top-3 contra un clúster real de Qdrant Cloud. Esta etapa implementa la recuperación documental de una arquitectura RAG, sin generación de respuestas ni decisiones sobre estudiantes.

## Alcance y fuentes
El corpus contiene tres **guías didácticas de elaboración propia**, no reglamentos oficiales de CUN. Se fundamentan conceptualmente en MEN (2016, 2023) y se identifican como propuestas. El dataset UCI 697 es una muestra portuguesa: su relación con las guías es temática a través del diccionario de variables. No se unen individuos portugueses con políticas colombianas ni se entrenan predictores.

## Preparación
Extraiga el proyecto completo. Instale `pip install -r requirements.txt`. Configure `QDRANT_URL` y `QDRANT_API_KEY` en un `.env` privado en la raíz, o como variables de entorno. La clave no se imprime ni se publica. En Colab puede inyectar los secretos en el entorno; también debe cargar las carpetas del proyecto. Las salidas guardadas se pueden leer sin credenciales.

In [5]:
from pathlib import Path
import sys, json, sqlite3
ROOT = Path.cwd()
if not (ROOT/'src/pipeline.py').exists(): ROOT = ROOT.parent
assert (ROOT/'src/pipeline.py').exists(), 'Abra el notebook con la carpeta completa del proyecto.'
sys.path.insert(0,str(ROOT/'src'))
from pipeline import *
print('Proyecto:',ROOT.name)

Proyecto: ACA1_Nube


## 1. Dataset y catálogo SQL
La tabla se audita sin modificar las categorías originales. SQLite constituye el catálogo SQL local de este prototipo; **el almacenamiento vectorial se despliega en cloud**. No se afirma que SQLite sea un servicio SQL remoto. El mapa temático relaciona variables del dataset con el tema del documento y se incorpora al payload vectorial.

In [6]:
audit = prepare_dataset()
display(pd.DataFrame([audit]))
with sqlite3.connect(ROOT/'data/catalogo.sqlite') as db:
    display(pd.read_sql_query('SELECT * FROM mapa_tematico',db))

,filas,columnas,faltantes,duplicados,pais,target
0,4424,37,0,0,Portugal,"{'Graduate': 2209, 'Dropout': 1421, 'Enrolled'..."


,tema,variable,descripcion
0,academico,Curricular units 1st sem (approved),Asignaturas aprobadas al cierre del primer sem...
1,financiero,Tuition fees up to date,Situación de pago de matrícula
2,financiero,Scholarship holder,Condición de becario
3,bienestar,Educational special needs,Necesidades educativas especiales


## 2. Ingesta y fragmentación
Se preservan archivo, título, posición, hash SHA-256 y origen. Para Markdown se utiliza página lógica 1, no paginación física.

Se eligieron ventanas de 60 palabras con solapamiento de 12 (20 %) para conservar contexto y reducir el riesgo de truncamiento del modelo. El solapamiento puede repetir información: Top-3 representa fragmentos, no necesariamente tres documentos distintos. El tamaño es una decisión de diseño y no se presenta como un óptimo experimental.

In [7]:
docs = chunks()
display(pd.DataFrame(docs).groupby(['file','tema']).size().rename('fragmentos').reset_index())
print('Fragmentos totales:',len(docs))
display(pd.DataFrame(docs)[['file','word_start','text']].head(3))

,file,tema,fragmentos
0,01_guia_academica.md,academico,8
1,02_guia_financiera.md,financiero,9
2,03_guia_bienestar.md,bienestar,9


Fragmentos totales: 26


,file,word_start,text
0,01_guia_academica.md,0,# Guía didáctica de acompañamiento académico #...
1,01_guia_academica.md,48,la gestión integral de la permanencia descrita...
2,01_guia_academica.md,96,con el docente. Una tutoría académica puede ce...


## 3. Embeddings abiertos en español
Se emplea `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` mediante FastEmbed (ONNX). Produce 384 dimensiones y se ejecuta en CPU, sin API generativa ni cuota por llamada. Es un modelo multilingüe de similitud: no se afirma que sea el mejor modelo de recuperación.

La guía permite modelos abiertos pero mezcla Gemini Embedding 2 con `RETRIEVAL_QUERY`. Según Google, Embedding 2 utiliza instrucciones de tarea, no ese parámetro. Aquí se documenta la elección alternativa permitida y se utiliza **el mismo modelo para documentos y preguntas**, evitando espacios vectoriales incompatibles (Google, s. f.).

In [8]:
encoder = TextEmbedding(MODEL)
example = list(encoder.embed(['acompañamiento académico']))[0]
print('Modelo:',MODEL,'| Dimensión:',len(example))
print('Configuración de tokenización:',encoder.model.tokenizer.truncation)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Modelo: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 | Dimensión: 384
Configuración de tokenización: {'max_length': 128, 'stride': 0, 'strategy': 'longest_first', 'direction': 'right'}


## 4. Conexión e indexación real en Qdrant Cloud
Distancia coseno, vectores normalizados, lotes de 32 e identificadores deterministas. Al repetir la carga no se duplican IDs ni se borra la colección. Si se cambia el corpus o el modelo, utilice una nueva versión de colección para evitar mezclar registros antiguos.

Credenciales exclusivamente desde el entorno o `.env`; se incluye solo `.env.example` sin secretos. El nivel gratuito es una instancia gestionada de un nodo: no se atribuyen características de alta disponibilidad o escalado serverless que no se verificaron.

In [9]:
client = connect()
print('Conexión remota correcta:',client.collection_exists(COLLECTION))
index_report = index(client,encoder,docs)
display(pd.DataFrame([index_report]))

Conexión remota correcta: True


,indexed,dimension,remote_count
0,26,384,26


## 5. Validación exploratoria con consultas predefinidas
Las seis consultas y su tema esperado se guardaron en `data/queries.json` antes de ejecutar la prueba. Se calcula coincidencia temática en Top-3 y hit por consulta. **No son juicios humanos de relevancia de cada fragmento**, ni una evaluación independiente: las preguntas fueron diseñadas para este corpus pequeño.

Se muestran las puntuaciones coseno, el texto, la fuente y los tiempos reales. Una similitud alta no prueba verdad, vigencia normativa o suficiencia de una orientación. No se establece un umbral universal de respuesta.

In [10]:
validation = validate(client,encoder)
summary = {k:v for k,v in validation.items() if k not in ['tests','limitations']}
display(pd.DataFrame([summary]))
rows=[]
for r in validation['tests']:
    for h in r['hits']:
        rows.append({'pregunta':r['question'],'rango':h['rank'],'coseno':round(h['score'],4),'tema':h['tema'],'archivo':h['file'],'fragmento':h['text']})
display(pd.DataFrame(rows))

,model,collection,mean_theme_precision_at_3,theme_hit_rate_at_3,median_total_ms,median_remote_ms
0,sentence-transformers/paraphrase-multilingual-...,aca1_permanencia_documental_v1,0.777778,1.0,188.13,170.15


,pregunta,rango,coseno,tema,archivo,fragmento
0,¿Qué puedo hacer para organizar el estudio sem...,1,0.7141,academico,01_guia_academica.md,"una consulta sobre hábitos de estudio, se prop..."
1,¿Qué puedo hacer para organizar el estudio sem...,2,0.5615,academico,01_guia_academica.md,con el docente. Una tutoría académica puede ce...
2,¿Qué puedo hacer para organizar el estudio sem...,3,0.5060,bienestar,03_guia_bienestar.md,El estudiante puede consultar la oferta dispon...
3,¿Cómo solicitar ayuda para comprender una asig...,1,0.6764,academico,01_guia_academica.md,la gestión integral de la permanencia descrita...
4,¿Cómo solicitar ayuda para comprender una asig...,2,0.5914,bienestar,03_guia_bienestar.md,documental debe quedar explícita. ## Inclusión...
5,¿Cómo solicitar ayuda para comprender una asig...,3,0.5884,academico,01_guia_academica.md,con el docente. Una tutoría académica puede ce...
6,¿Qué debo consultar si no puedo pagar la matrí...,1,0.3072,financiero,02_guia_financiera.md,alternativas vigentes. Antes de asumir una obl...
7,¿Qué debo consultar si no puedo pagar la matrí...,2,0.3016,financiero,02_guia_financiera.md,entre una beca y graduarse. ## Uso en el busca...
8,¿Qué debo consultar si no puedo pagar la matrí...,3,0.2994,financiero,02_guia_financiera.md,compromiso de una universidad. La consideració...
9,¿Cómo preparar una solicitud de beca?,1,0.5580,financiero,02_guia_financiera.md,información oficial y suficiente. ## Registro ...


## 6. Consulta en vivo
Modifique `pregunta` y ejecute esta celda para la demostración. Esta función consulta el clúster remoto, no una tabla de resultados guardados.

In [18]:
#@title Consulta en vivo
pregunta = "¿Dónde solicitar acompañamiento psicosocial?" #@param {type:"string"}
from IPython.display import display, HTML
from html import escape
resultado = search(client, encoder, pregunta, k=3)
display(HTML("<h3>Resultados de la consulta</h3><p><b>Pregunta:</b> "+escape(pregunta)+"</p>"))
for h in resultado['hits']:
    contenido = "<div style='padding:14px;margin:12px 0;border:1px solid #999;border-radius:6px'>"
    contenido += "<b>"+str(h['rank'])+". "+escape(h['document'])+"</b>"
    contenido += "<p>Similitud coseno: "+format(h['score'],'.4f')+"</p>"
    contenido += "<p>"+escape(h['text'])+"</p>"
    contenido += "<a href='"+escape(h['source'],quote=True)+"' target='_blank'>Abrir documento de origen</a></div>"
    display(HTML(contenido))
print("Consulta real a Qdrant. Tiempo total:", resultado['total_ms'], "ms")
print("La similitud mide cercanía de significado; no es una probabilidad de verdad.")


Consulta real a Qdrant. Tiempo total: 601.35 ms
La similitud mide cercanía de significado; no es una probabilidad de verdad.


## 7. Pruebas de límites y conclusiones
Se verifica el rechazo de preguntas vacías y la coherencia del filtro temático. Una consulta fuera del corpus seguirá devolviendo vecinos: por ello, el sistema no debe afirmar que siempre encuentra una respuesta válida. El prototipo no genera recomendaciones automáticas ni decide sobre elegibilidad académica o financiera.

In [12]:
try:
    search(client,encoder,'')
except ValueError:
    print('Entrada vacía rechazada correctamente.')
filtered = search(client,encoder,'¿Cómo solicitar una beca?',tema='financiero')
assert all(h['tema']=='financiero' for h in filtered['hits'])
assert index_report['remote_count']==len(docs)
print('Filtro temático y conteo remoto verificados.')
print(validation['limitations'])

Entrada vacía rechazada correctamente.
Filtro temático y conteo remoto verificados.
Prueba exploratoria por tema preasignado; no mide relevancia humana de cada fragmento, ni precisión general. Similitud no equivale a probabilidad de verdad.


## Referencias
Google. (s. f.). *Embeddings*. Google AI for Developers. https://ai.google.dev/gemini-api/docs/embeddings

Ministerio de Educación Nacional. (2016). *Lineamientos de política de bienestar para instituciones de educación superior*. https://www.mineducacion.gov.co/1759/articles-360314_recurso.pdf

Ministerio de Educación Nacional. (2023, 29 de mayo). *Gestión de la permanencia*. https://www.mineducacion.gov.co/sistemasinfo/spadies/secciones/Gestion-de-la-permanencia/

Qdrant. (s. f.). *Create a cluster*. https://qdrant.tech/documentation/cloud/create-cluster/

Realinho, V., Vieira Martins, M., Machado, J., & Baptista, L. (2021). *Predict students’ dropout and academic success* [Conjunto de datos]. UCI Machine Learning Repository. https://doi.org/10.24432/C5MC89

Reimers, N., & Gurevych, I. (2019). Sentence-BERT: Sentence embeddings using Siamese BERT-networks. En *Proceedings of EMNLP-IJCNLP* (pp. 3982-3992). https://doi.org/10.18653/v1/D19-1410

Sentence Transformers. (s. f.). *paraphrase-multilingual-MiniLM-L12-v2* [Modelo]. Hugging Face. https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2

**Corpus:** Cometa Hernández, O. S. (2026). *Guías didácticas de acompañamiento académico, orientación financiera y bienestar e inclusión* [Corpus de elaboración propia para ACA 1].